# 00 — Smoke Test

Read the OHLCV and FRED Parquet layers via Spark and verify basic shape, schema, and date coverage.

In [1]:
import sys
sys.path.insert(0, '..')
from src.spark_session import get_spark

spark = get_spark('smoke_test')
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/20 05:09:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## OHLCV

In [2]:
ohlcv = spark.read.parquet('../data/parquet/ohlcv')
print('total rows:', ohlcv.count())
print('distinct tickers:', ohlcv.select('ticker').distinct().count())

total rows: 1940547


distinct tickers: 503


In [3]:
ohlcv.printSchema()

root
 |-- date: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- adj_close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- ticker: string (nullable = true)



In [4]:
from pyspark.sql import functions as F
ohlcv.agg(F.min('date').alias('min_date'), F.max('date').alias('max_date')).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2010-01-04|2026-04-17|
+----------+----------+



In [5]:
ohlcv.orderBy('ticker','date').show(5, truncate=False)

+----------+------------------+------------------+------------------+------------------+------------------+-------+------+
|date      |open              |high              |low               |close             |adj_close         |volume |ticker|
+----------+------------------+------------------+------------------+------------------+------------------+-------+------+
|2010-01-04|22.45350456237793 |22.625179290771484|22.26752471923828 |22.389127731323242|19.810985565185547|3815561|A     |
|2010-01-05|22.324750900268555|22.3319034576416  |22.00286102294922 |22.145923614501953|19.595775604248047|4186031|A     |
|2010-01-06|22.06723976135254 |22.174535751342773|22.00286102294922 |22.06723976135254 |19.52616310119629 |3243779|A     |
|2010-01-07|22.017166137695312|22.045780181884766|21.81688117980957 |22.03862762451172 |19.50084686279297 |3095172|A     |
|2010-01-08|21.917024612426758|22.06723976135254 |21.745351791381836|22.03147315979004 |19.49452018737793 |3733918|A     |
+----------+----

## FRED macro

In [6]:
fred = spark.read.parquet('../data/parquet/fred_macro')
print('rows:', fred.count())
fred.printSchema()
fred.agg(F.min('date').alias('min_date'), F.max('date').alias('max_date')).show()
fred.orderBy('date').show(5, truncate=False)

rows: 5954
root
 |-- date: date (nullable = true)
 |-- VIXCLS: double (nullable = true)
 |-- DGS10: double (nullable = true)
 |-- FEDFUNDS: double (nullable = true)
 |-- CPIAUCSL: double (nullable = true)
 |-- UNRATE: double (nullable = true)

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2010-01-01|2026-04-20|
+----------+----------+

+----------+------+-----+--------+--------+------+
|date      |VIXCLS|DGS10|FEDFUNDS|CPIAUCSL|UNRATE|
+----------+------+-----+--------+--------+------+
|2010-01-01|NULL  |NULL |0.11    |217.488 |9.8   |
|2010-01-02|NULL  |NULL |0.11    |217.488 |9.8   |
|2010-01-03|NULL  |NULL |0.11    |217.488 |9.8   |
|2010-01-04|20.04 |3.85 |0.11    |217.488 |9.8   |
|2010-01-05|19.35 |3.77 |0.11    |217.488 |9.8   |
+----------+------+-----+--------+--------+------+
only showing top 5 rows



In [7]:
spark.stop()